# Swish / SiLU — Searching for Activation Functions (2017)

_Papers · Foundations & Optimization_

**Replace ReLU's hard hinge with a smooth self-gated curve, $x\cdot\sigma(\beta x)$, that dips slightly below zero — found by automated search and matching or beating ReLU on deep nets.**

---

This notebook builds Swish from scratch, one piece at a time. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Define Swish and check it against PyTorch's SiLU

Swish is **self-gated**: it multiplies the input $x$ by a sigmoid gate $\sigma(\beta x)$. At $\beta=1$ this is exactly what PyTorch calls SiLU, so we can use `F.silu` as an oracle to confirm our from-scratch version is correct.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

def my_swish(x, beta=1.0):
    """Swish from scratch — Ramachandran, Zoph & Le (2017), Section 4: f(x)=x*sigmoid(beta*x).
       At beta=1 this is SiLU (PyTorch's F.silu)."""
    return x * torch.sigmoid(beta * x)          # self-gated: x times its own sigmoid gate

# ---- THE ORACLE: my Swish at beta=1 must equal F.silu ----
x = torch.randn(1000)
ok = torch.allclose(my_swish(x, beta=1.0), F.silu(x), atol=1e-6)
print("allclose vs F.silu (beta=1):", ok)                     # expect True

### Step 2 — Verify the derivative against autograd

The paper gives a closed form for Swish's derivative: $f'(x)=\beta f(x) + \sigma(\beta x)\,(1 - \beta f(x))$. We compute it by hand and compare it to PyTorch's automatic differentiation. We also check the slope at a *negative* input — Swish has a nonzero gradient there, whereas ReLU's gradient is flat zero, which is why ReLU units can 'die'.

In [ ]:
# ---- the derivative (Section 4) checked against autograd ----
xb = torch.randn(5, requires_grad=True)
y = my_swish(xb, beta=1.0).sum()
y.backward()
auto = xb.grad

s = torch.sigmoid(xb)
f = xb * s
paper = 1.0 * f + s * (1.0 - 1.0 * f)          # f'(x)=beta*f + sigma(beta x)(1 - beta f)
print("derivative matches autograd:", torch.allclose(auto, paper, atol=1e-6))  # True

# ---- nonzero gradient at a negative input (ReLU's is zero) ----
xn = torch.tensor([-1.0], requires_grad=True)
my_swish(xn).backward()
print("Swish slope at x=-1:", round(xn.grad.item(), 4), "(ReLU's slope there is 0.0)")

### Step 3 — Recompute the worked example

Plug a few values through $f(x)=x\cdot\sigma(x)$ to see the numbers from the lesson. Notice that negative inputs give small *negative* outputs (the non-monotonic 'bump'), not zero like ReLU.

In [ ]:
# ---- recompute the worked example (beta=1) ----
for xv in [1.0, -1.0, 2.0, -3.0]:
    print(f"swish({xv:+.0f}) =", round(my_swish(torch.tensor(xv)).item(), 4))
# expect 0.7311, -0.2689, 1.7616, -0.1423

### Step 4 — Ablation: Swish vs ReLU on a tiny net

Train the *same* small network twice — once with ReLU, once with SiLU (Swish at $\beta=1$) — on the same nonlinear regression target, with identical initialization. Because everything but the activation is held fixed, the final-loss gap isolates the effect of the activation. On deeper/smaller nets Swish often matches or edges out ReLU, matching the paper's qualitative claim.

In [ ]:
# ---- ABLATION: Swish vs ReLU on a tiny net (same data, same init seed) ----
def make_net(act):
    return nn.Sequential(nn.Linear(20, 64), act, nn.Linear(64, 64), act, nn.Linear(64, 1))

X = torch.randn(256, 20)
w_true = torch.randn(20, 1)
Y = (X @ w_true).pow(2).sum(1, keepdim=True) + 0.1 * torch.randn(256, 1)  # nonlinear target

def train(act, steps=300):
    torch.manual_seed(0)                         # identical init for a fair comparison
    net = make_net(act)
    opt = torch.optim.Adam(net.parameters(), lr=1e-2)
    for _ in range(steps):
        opt.zero_grad()
        loss = F.mse_loss(net(X), Y)
        loss.backward()
        opt.step()
    return loss.item()

print("final loss  ReLU :", round(train(nn.ReLU()),  4))
print("final loss  SiLU :", round(train(nn.SiLU()),  4))   # Swish(beta=1); often matches/edges ReLU

## Visualize the data & results

_What does the Swish curve look like next to ReLU? Is it smooth, and does it really dip below zero (the non-monotonic 'bump') for negative inputs?_

### Step 1 — Tabulate Swish and ReLU side by side

Reimplement Swish and ReLU in plain NumPy and evaluate them on a grid of inputs from $-5$ to $5$. Printing the two columns together makes the contrast clear: ReLU is flat zero for all negatives, while Swish curves smoothly and dips slightly below zero.

In [ ]:
import numpy as np

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def swish(x, beta=1.0):
    return x * sigmoid(beta * x)

def relu(x):
    return np.maximum(0.0, x)

xs = np.arange(-5.0, 5.001, 0.5)        # 21 points
print("x     :", np.round(xs, 1).tolist())
print("swish :", np.round(swish(xs), 4).tolist())
print("relu  :", np.round(relu(xs),  4).tolist())

### Step 2 — Locate the non-monotonic 'bump'

Swish is *non-monotonic*: as $x$ goes negative it first dips down before rising back toward zero. We scan a fine grid on the negative side to find the exact minimum — the lowest point of that bump — which sits near $x\approx-1.28$ with value $\approx-0.28$.

In [ ]:
# locate the non-monotonic minimum (the 'bump')
fine = np.linspace(-3, 0, 200000)
ys = swish(fine)
i = ys.argmin()
print("Swish minimum: x =", round(float(fine[i]), 4), " value =", round(float(ys[i]), 4))
# expect x ~ -1.2785, value ~ -0.2785  -> this is the non-monotonic bump

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Compute Swish at $x=-2$ with $\beta=1$ (use $\sigma(-2)=0.1192$). Then say how this differs from ReLU at the same input.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Gate: $\sigma(\beta x)=\sigma(-2)=0.1192$. — _The sigmoid of $\beta x$ is the self-gate; it is small but nonzero for a moderately negative input._
- Output: $f(-2)=x\cdot\sigma(\beta x)=-2\times 0.1192=-0.2384$. — _Swish multiplies the raw $x$ by its gate._
- Compare to ReLU: $\max(0,-2)=0$, with gradient $0$ there. — _ReLU zeros out all negatives and has no gradient on that side._

**Answer:** Swish gives $-0.2384$ (a small negative, part of the bump), while ReLU gives exactly $0$. Swish also has a nonzero slope here, so the unit can keep learning where a ReLU unit would be 'dead'.

</details>

**Problem 2.** Beta effect: evaluate Swish at $x=2$ for $\beta=0$, a small $\beta=0.1$, and a large $\beta=5$, and relate each to a limiting function. ($\sigma(0)=0.5$, $\sigma(0.2)=0.5498$, $\sigma(10)=0.99995$.)

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- $\beta=0$: $f(2)=2\cdot\sigma(0)=2\cdot 0.5=1.0$. — _The gate is stuck at $0.5$, so Swish is the line $x/2$ — $2/2=1$ (paper: $\beta=0\Rightarrow x/2$)._
- $\beta=0.1$: $f(2)=2\cdot\sigma(0.2)=2\cdot 0.5498=1.0997$. — _Small $\beta$ keeps the gate close to $0.5$, so the output stays near the $x/2$ line._
- $\beta=5$: $f(2)=2\cdot\sigma(10)=2\cdot 0.99995=1.9999$. — _Large $\beta$ drives the gate to $\approx 1$, so $f(2)\approx 2=\mathrm{ReLU}(2)$ (paper: $\beta\to\infty\Rightarrow$ ReLU)._

**Answer:** As $\beta$ grows from $0$ to large, Swish at $x=2$ moves from $1.0$ (the line $x/2$) through $1.0997$ up to $1.9999$ (ReLU's $2$). So $\beta$ tunes Swish smoothly between a straight line and ReLU.

</details>

**Problem 3.** Ablation: in a small network you train once with ReLU and once with Swish, everything else fixed. The CODEVIZ reports Swish reaching slightly lower final loss. Why might allowing small negative outputs and a smooth corner help, and why is this our number rather than the paper's?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note ReLU zeros every negative pre-activation and has zero gradient there, so some units can get stuck ('die'). — _No gradient means no learning signal for those units._
- Note Swish keeps a small nonzero value and slope for negatives, and is smooth at $0$. — _Gradients keep flowing through near-zero and negative regions, and the smooth shape can ease optimization._
- Mark the result as OUR small run, not the paper's reported metric. — _The spec forbids presenting our toy-run numbers as the paper's headline; the paper's own gains are quoted separately in 'results'._

**Answer:** Swish's nonzero negative gradient and smoothness give the optimizer signal where ReLU gives none, which can edge out a lower loss on deep/small nets — matching the paper's qualitative claim that Swish 'tends to work better than ReLU on deeper models'. The exact loss gap in CODEVIZ is our own small-scale run, not the paper's reported number.

</details>